# V2 Phase 11 — Colab GPU live artefact (`llama_cpp`)

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

Validates the same live runner used by `app/streamlit_app.py` on a **fresh** question:
- Single-Agent RAG
- Multi-Agent RAG
- Uncertainty / Abstention RAG

Uses **llama_cpp + Qwen3-8B**, not mock. Does **not** run the 140-question benchmark. Does **not** start Phase 12.

## Setup

Push latest V2 (including Phase 11) to branch `cursor/empty-v2-workspace`, then run all cells.

Requires Phase 8 KB on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/`.

**Outputs:** `results/config/phase11_smoke_test.json`, `phase11_live_smoke.json`

**Manual browser demo:** section 8 starts Streamlit on this GPU runtime and prints a temporary public URL. Do not use mock. Do not run the 140-question benchmark.

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'cursor/empty-v2-workspace'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'smoke_live_artefact.py').is_file():
    raise FileNotFoundError(f'Phase 11 script missing at {V2_ROOT}. Push Phase 11 to GitHub first.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Restore knowledge base from Drive (or rebuild)

Does **not** copy the Mac Chroma database. Reuses the Colab-built index from Phase 8 when available.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
restored = True

for rel in ('artifacts/knowledge_base/index', 'artifacts/knowledge_base/documents'):
    src = DRIVE_ROOT / rel
    dst = V2 / 'knowledge_base' / rel.split('/')[-1]
    if not src.is_dir():
        print('Missing on Drive:', src)
        restored = False
        break
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

if not restored:
    print('Drive KB not found — falling back to build_index.py (Option B)...')
    !PYTHONPATH=. python scripts/build_index.py --distractors 50

## 4. Index preflight validation

In [ ]:
!PYTHONPATH=. python scripts/validate_kb_index.py

## 5. Phase 11 live comparison (`llama_cpp`, one fresh question)

Same `run_live_comparison()` used by Streamlit. Three architectures, independently, no chaining.

In [ ]:
!PYTHONPATH=. python scripts/smoke_live_artefact.py --backend llama_cpp --fresh-only

## 6. Check UI-equivalent fields

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase11_runtime_fingerprint.json')
smoke = Path('results/config/phase11_smoke_test.json')
detail = Path('results/config/phase11_live_smoke.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
print('detail:', detail.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', data.get('actual'))
    env = data.get('environment') or {}
    print('device:', env.get('device'))
    print('gpu:', (env.get('gpu') or {}).get('name'))
    print('backend:', (env.get('model_config') or {}).get('backend'))
if detail.is_file():
    detail_data = json.loads(detail.read_text())
    print('backend:', detail_data.get('backend'))
    for comparison in detail_data.get('comparisons', []):
        print('---')
        print('source:', comparison.get('question_source'), 'qid:', comparison.get('question_id'))
        print('question:', (comparison.get('question') or '')[:160])
        for architecture, case in (comparison.get('results') or {}).items():
            vr = case.get('verification_result') or {}
            print(
                architecture,
                'n_evidence=', len(case.get('retrieved_evidence') or []),
                'answer_len=', len(case.get('answer') or ''),
                'verify=', vr.get('verification_score'),
                'confidence=', case.get('confidence'),
                'threshold=', case.get('threshold'),
                'decision=', case.get('decision'),
                'error=', case.get('error'),
            )

## 7. Save Phase 11 Colab results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase11')
dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase11_runtime_fingerprint.json',
    'phase11_smoke_test.json',
    'phase11_live_smoke.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, dest / name)
        print('copied', name)
print('Done:', dest)

## 8. Manual Streamlit live demo (browser URL)

This section starts the **existing** `app/streamlit_app.py` on this Colab GPU runtime and exposes it through a temporary Cloudflare tunnel.

- Backend: **llama_cpp / Qwen3-8B** (`V2_LIVE_BACKEND=llama_cpp`). Do **not** select mock.
- Knowledge base: existing Phase 6 / Colab-restored Chroma index (no rebuild unless preflight already failed).
- Enter a **fresh question** in the browser and run all three architectures.
- Do **not** start the 140-question benchmark.
- Leave section 9 running while you test. Run section 10 only after the manual test.

In [ ]:
from pathlib import Path
import os
import socket
import sys
import time

import torch

V2 = Path('/content/capstone-rag/V2')
os.chdir(V2)
sys.path.insert(0, str(V2))

print('=== 1. GPU ===')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Runtime → Change runtime type → GPU (T4).')
gpu_name = torch.cuda.get_device_name(0)
print('cuda:', True)
print('gpu:', gpu_name)
if 'T4' not in gpu_name and 'Tesla' not in gpu_name:
    print('NOTE: expected Tesla T4; recorded GPU name above.')

print('=== 2. V2 repository ===')
app_path = V2 / 'app' / 'streamlit_app.py'
if not app_path.is_file():
    raise FileNotFoundError(f'Streamlit app missing: {app_path}. Push Phase 11 and re-run cell 1.')
print('app:', app_path)
print('cwd:', Path.cwd())

print('=== 3. Chroma index ===')
from src.config import get_path, load_experiment_config, project_root
from src.retrieval.index import COLLECTION_NAME
from src.retrieval.preflight import validate_index_preflight

config = load_experiment_config()
retrieval_cfg = config.section('retrieval')
index_dir = get_path(config, 'kb_index')
manifest = (project_root() / str(retrieval_cfg.get('index_manifest') or 'knowledge_base/index/index_manifest.json')).resolve()
preflight = validate_index_preflight(
    index_dir,
    manifest_path=manifest,
    collection_name=str(retrieval_cfg.get('collection_name') or COLLECTION_NAME),
)
print(preflight)
if int(preflight.get('actual_count') or 0) <= 0:
    raise RuntimeError('Chroma index is empty. Restore the Phase 8 KB from Drive or rebuild before the live demo.')
print('PREFLIGHT PASS')

In [ ]:
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import os
import re
import socket
import subprocess
import sys
import time
from pathlib import Path

V2 = Path('/content/capstone-rag/V2')
os.chdir(V2)

def _wait_port(port: int, timeout: int = 90) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        sock = socket.socket()
        try:
            sock.connect(('127.0.0.1', port))
            return True
        except OSError:
            time.sleep(1)
        finally:
            sock.close()
    return False

os.system("pkill -f 'streamlit run app/streamlit_app.py' || true")
os.system('pkill -f "cloudflared tunnel --url" || true')
time.sleep(2)

env = os.environ.copy()
env['PYTHONPATH'] = str(V2)
env['V2_LIVE_BACKEND'] = 'llama_cpp'

st_log = Path('/tmp/v2_streamlit.log')
cf_log = Path('/tmp/v2_cloudflared.log')
st_handle = st_log.open('w')
cf_handle = cf_log.open('w')

st_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', 'app/streamlit_app.py',
        '--server.port=8501',
        '--server.address=0.0.0.0',
        '--server.headless=true',
        '--browser.gatherUsageStats=false',
    ],
    cwd=str(V2),
    env=env,
    stdout=st_handle,
    stderr=subprocess.STDOUT,
)
print('Streamlit pid:', st_proc.pid)
if not _wait_port(8501):
    print(st_log.read_text()[-3000:])
    raise RuntimeError('Streamlit did not bind 127.0.0.1:8501')
print('Streamlit is listening on http://127.0.0.1:8501')

cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'],
    stdout=cf_handle,
    stderr=subprocess.STDOUT,
)
print('cloudflared pid:', cf_proc.pid)

url = None
for _ in range(45):
    time.sleep(1)
    text = cf_log.read_text(errors='replace')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
    if match:
        url = match.group(0)
        break

if not url:
    print(cf_log.read_text(errors='replace')[-3000:])
    raise RuntimeError('Temporary tunnel URL not found. Re-run this cell.')

Path('/tmp/v2_live_demo_url.txt').write_text(url + '\n', encoding='utf-8')
print('=' * 72)
print('BROWSER URL — open this in your browser:')
print(url)
print('=' * 72)
print('Sidebar backend must be llama_cpp (preselected). Do not use mock.')
print('Enter a fresh question → Run all three architectures.')
print('Expected UI: evidence, scores/metadata, answer, verification, confidence, threshold, ANSWER/ABSTAIN, or ERROR/UNAVAILABLE.')
print('Leave the next cell running while you test. First Qwen3 load on T4 can take a few minutes.')

## 9. Keep Streamlit alive

Run this cell and **leave it running** while you use the browser URL.

Interrupt this cell only when the manual live test is finished, then run section 10.

In [ ]:
import socket
import time
from pathlib import Path

url_file = Path('/tmp/v2_live_demo_url.txt')
url = url_file.read_text(encoding='utf-8').strip() if url_file.is_file() else '(URL cell not run)'
print('BROWSER URL:', url)
print('Keep this cell running. Interrupt when the manual test is done.')

while True:
    sock = socket.socket()
    try:
        sock.connect(('127.0.0.1', 8501))
        alive = True
    except OSError:
        alive = False
    finally:
        sock.close()
    if not alive:
        raise RuntimeError('Streamlit is no longer listening on :8501. Re-run section 8.')
    time.sleep(30)

## 10. After the manual browser test — save live-session record

Run this cell **only after** you entered a fresh question in the browser and ran all three architectures.

Set `MANUAL_TEST_COMPLETED = True` and fill the observed fields. Do not mark PASS unless you actually saw the UI results. This cell does **not** invent answers.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from shutil import copy2

MANUAL_TEST_COMPLETED = False  # set True only after the browser test
FRESH_QUESTION_USED = ''  # paste the fresh question you typed
OBSERVED_DECISIONS = {
    'single_agent': '',      # ANSWER | ERROR | UNAVAILABLE
    'multi_agent': '',
    'multi_agent_uq': '',    # ANSWER | ABSTAIN | ERROR | UNAVAILABLE
}
N_EVIDENCE = {
    'single_agent': None,
    'multi_agent': None,
    'multi_agent_uq': None,
}
SAW_FIELDS = {
    'retrieved_evidence': False,
    'retrieval_scores_metadata': False,
    'generated_answer': False,
    'verification': False,
    'confidence': False,
    'threshold': False,
    'decision': False,
}
NOTES = ''

if not MANUAL_TEST_COMPLETED:
    raise RuntimeError('Set MANUAL_TEST_COMPLETED = True only after you finished the browser live test.')

url_file = Path('/tmp/v2_live_demo_url.txt')
record = {
    'phase': 11,
    'test_name': 'phase11_colab_streamlit_manual_live',
    'command': 'notebooks/colab_phase11_live.ipynb section 8–10; streamlit run app/streamlit_app.py; V2_LIVE_BACKEND=llama_cpp',
    'backend': 'llama_cpp',
    'mock_used': False,
    'browser_url_file': str(url_file) if url_file.is_file() else None,
    'fresh_question_used': FRESH_QUESTION_USED,
    'observed_decisions': OBSERVED_DECISIONS,
    'n_evidence': N_EVIDENCE,
    'saw_fields': SAW_FIELDS,
    'notes': NOTES,
    'status': 'PASS' if all(SAW_FIELDS.values()) and FRESH_QUESTION_USED.strip() else 'NEEDS VERIFICATION',
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
}
out = Path('/content/capstone-rag/V2/results/config/phase11_colab_live_demo.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
print('Wrote', out)
print(json.dumps(record, indent=2))

drive_dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase11')
if drive_dest.parent.is_dir():
    drive_dest.mkdir(parents=True, exist_ok=True)
    copy2(out, drive_dest / out.name)
    print('Copied to', drive_dest / out.name)
else:
    print('Drive folder not mounted; local record only.')

print('Copy phase11_colab_live_demo.json into local V2/results/config/ then ask for evidence/master-record update.')